In [ ]:
import warnings
import matplotlib.gridspec as gridspec
import rioxarray # for the extension to load
import numpy as np
import rioxarray as rxr
import rasterio
import xarray as xr
from tools import load_ana_data, format_lat_lon_ticks, plot_scale_bar, scaloa
from glob import glob
import hvplot.xarray
import hvplot.pandas
import holoviews as hv
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from tools import lonlim, latlim, process_satellite_image, standardize_lat_lon, rasterize_geodataframe
from shapely.geometry import box, Point, shape
import geopandas as gpd
import fiona
from xhistogram.xarray import histogram
import cmasher as cmr
from matplotlib.colors import ListedColormap
from matplotlib.colors import Normalize
import gsw

fiona.drvsupport.supported_drivers['libkml'] = 'rw' # enable KML support which is disabled by default
fiona.drvsupport.supported_drivers['LIBKML'] = 'rw' # enable KML support which is disabled by default

In [ ]:
fnames = glob("../data/external/hydroweb/5-*")

ana = []
station = []
for fname in tqdm(fnames):
    ana.append(load_ana_data(fname))
    station.append(fname.split("-")[-4])
ana = xr.concat(ana, "station").assign_coords(station=station)

In [ ]:
# Your longitude and latitude
longitude = ana.longitude.values
latitude = ana.latitude.values

# Create a DataFrame for your point
df = pd.DataFrame({"longitude": longitude, "latitude": latitude})

In [ ]:
dx = dy = 30
dlon = dx/(111.2e3*np.cos(latitude.mean()*np.pi/180))
dlat = dy/(111.2e3)

fnames = glob(f"../data/external/swot/*")
fnames.sort()
fnames = np.array(fnames)


variables = [
    "height", "water_frac", "classification", "geoid", 
    "illumination_time", "pixel_area", 
    "pole_tide", "load_tide_fes", "solid_earth_tide", 
    "height_cor_xover", "model_dry_tropo_cor", 
    "model_wet_tropo_cor", "iono_cor_gim_ka"
]

ds = {k:[] for k in ana.station.values}
heights = {k:[] for k in ana.station.values}
for fname in tqdm(fnames):
    swot = xr.open_dataset(fname, group="pixel_cloud")[variables]
    swot.load()

    npass = int(fname.split("_")[-6])
    for lon, lat, station in zip(ana.longitude.values, ana.latitude.values, ana.station.values):

        where = (
            (swot.longitude>lon-dlon)&(swot.longitude<lon+dlon)&
            (swot.latitude>lat-dlat)&(swot.latitude<lat+dlat)&
            (swot.classification>2)&(swot.classification<6)&
            (swot.water_frac>0.1)&(~np.isnan(swot.height))
        ).values
        
        ind = np.argwhere(where).ravel()
        dsi = swot.sel(points=ind)

        # The reported height is after correcting the measured range
        # for instrument calibrations and signal propagation delays due to the ionosphere and
        # the dry and wet components of the troposphere, as well as crossover calibration
        # (height_cor_xover). These applied corrections affect the estimation of the height of
        # the Earth surface with respect to an Earth-centered, Earth-fixed frame. Model-based
        # estimates of tides (solid Earth, load, and pole) and geoid variations are provided in
        # this product, but they have not been applied to the reported value of height.
        # https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-docs/web-misc/swot_mission_docs/pdd/D-56411_SWOT_Product_Description_L2_HR_PIXC_20240913.pdf
        correction = (
                dsi.pole_tide +
                dsi.load_tide_fes +
                dsi.solid_earth_tide +
                dsi.geoid
            )
        
        dsi["water_level"] = (dsi.height-correction)

        if dsi.points.size>0:
            dsa = dsi.median("points").assign_coords(time=dsi.illumination_time.mean().dt.round("h"), station=station).expand_dims(["time", "station"])
            ds[station].append(dsa)
        
        if dsi.points.size>1:

            level = xr.merge([
                dsi.illumination_time.mean().dt.round("min").rename("illumination_time"),
                dsi["water_level"].median(),
            ])

            level["time"] = level.illumination_time
            level = level.set_coords("time").drop_vars("illumination_time").expand_dims("time")

            level = level.assign_coords(station=station).expand_dims("station")

            level["pass"] = npass
            
            heights[station].append(level)

ana_swot = xr.concat([xr.concat(heights[k], "time").drop_duplicates("time") for k in ana.station.values], "station")
ana_swot = ana_swot.isel(time=np.argsort(ana_swot.time.values))
ana_swot = ana_swot.dropna("time", how="all")
ana_swot["time"] = ana_swot["time"] - np.timedelta64(3, "h")
ana_swot["time"].attrs["timezone"] = "UTC-3"

correction = (ana.height.interp(time=ana_swot.time)-ana_swot.water_level).median()
ana_swot["water_level"] += correction
ana_swot.attrs["correction (m)"] = correction.values
ana_swot["water_level"].attrs = {"units": "m", "long_name": "height above geoid"}

ana_swot.to_netcdf("../data/processed/swot_ana.nc")
ana.to_netcdf("../data/processed/ana.nc")

ds = xr.concat([xr.concat(ds[k], "time").drop_duplicates("time") for k in ana.station.values], "station")

In [ ]:
correction = (
        ds.pole_tide +
        ds.load_tide_fes +
        ds.solid_earth_tide +
        ds.geoid
    )

variables = [
    "height", "geoid", 
    "pole_tide", "load_tide_fes", "solid_earth_tide", 
]

fig, ax = plt.subplots(2,1,figsize=(8,6))

for a,k in zip(ax, ana.station.values):
    (ds.height-correction).sel(station=k).dropna("time").plot(ax=a, label="corrected height", marker="o", markersize=4)
    for var in variables:
        ds.sel(station=k)[var].dropna("time").plot(ax=a, label=var, marker="o", markersize=4, ls="--")

    ana.sel(station=k).height.plot(ax=a, color="k", label="in situ data")
    a.grid(True, linestyle="--", alpha=0.5)
    a.set(title=k)
ax[0].legend(bbox_to_anchor=(1.05, 0.5), loc='upper left', borderaxespad=0.)
ax[0].set(xlabel="", xticklabels=[]);